# 🚀 NanoStream-OD 2.0: High-Quality Face Detector Training on Google Colab (T4 GPU)

This notebook trains **NanoStream-OD 2.0 (High-Quality Ultra-Efficient Face Detector)** on a large-scale real face dataset (WIDER Face / Curated Diversity Dataset) using a **T4 GPU** in under 8 minutes.

- **Target:** Production Real-Time Face Detection on Microcontrollers (<50KB SRAM) & Standard Laptops
- **Architecture:** Depthwise Inverted Residuals + Dual-Scale Zero-NMS Heads + 3x3 Local Peak Maxpooling
- **Output:** Trained PyTorch checkpoint (`.pt`) + Static Bit-Exact C Header (`model_weights.h`)

## 📦 Step 1: Environment Setup & Clone Repository

In [ ]:
# Check GPU
!nvidia-smi

# Clone repo or install dependencies
!git clone https://github.com/Flaxmbot/nanostream.git || true
%cd nanostream
!pip install -q torch torchvision opencv-python pytest

## 📥 Step 2: Download & Prepare Large-Scale Face Dataset
Downloads high-diversity real face dataset with annotations and photorealistic augmentations.

In [ ]:
import os, urllib.request, zipfile, tarfile
import torch, cv2, numpy as np

# Ensure dataset directory exists
os.makedirs("data/real_faces/images", exist_ok=True)

# Download online high-diversity portraits (or generate 2,000+ auto-labeled crops)
print("Downloading diverse public-domain training faces...")
from nanostream.dataset import download_online_face_dataset
download_online_face_dataset("data/real_faces")

print("Dataset ready for GPU training!")

## 🧠 Step 3: Train High-Quality NanoStream-OD on GPU
Trains for 5,000 steps with batch size 32, mixed precision (AMP), Cosine Annealing, and multi-scale auxiliary supervision.

In [ ]:
import torch
from nanostream.config import NanoStreamConfig
from nanostream.model import NanoStreamOD
from nanostream.dataset import WebcamFaceDataset
from nanostream.train_faces import evaluate_real_recall
from nanostream.head import detection_loss
from nanostream.data import collate
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")

# High-Quality Model Config (Dual-scale, 48-channel backbone, high receptive field)
cfg = NanoStreamConfig(
    input_size=160,
    in_channels=1,
    stage_widths=(16, 24, 32, 48),
    head_hidden=48,
    dual_scale=True,
    num_classes=1
)

model = NanoStreamOD(cfg).to(device)
print(f"Model parameters: {model.param_count():,} ({model.param_count()*2/1024:.1f} KB @ int16)")

# Dataset & DataLoader
train_ds = WebcamFaceDataset("data/real_faces", img_size=160, augment=True, cache_in_ram=True)
eval_ds = WebcamFaceDataset("data/real_faces", img_size=160, augment=False, cache_in_ram=True)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate, num_workers=2, pin_memory=True)

opt = torch.optim.AdamW(model.parameters(), lr=0.003, weight_decay=1e-4)
steps = 5000
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps, eta_min=1e-5)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

print("Starting GPU Training...")
step = 0
best_recall = 0.0
model.train()

while step < steps:
    for imgs, tgts in train_loader:
        if step >= steps: break
        imgs = imgs.to(device)
        tgts = [{k: v.to(device) for k, v in t.items()} for t in tgts]

        opt.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            preds = model(imgs)
            losses = detection_loss(preds, tgts, w_box=3.0, w_l1=1.0, w_obj=1.5)
            loss = losses["total"]

        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(opt)
        scaler.update()
        scheduler.step()

        if (step + 1) % 250 == 0 or step == steps - 1:
            recall, prec, h, t = evaluate_real_recall(model, eval_ds, n=80, conf=0.30)
            lr = opt.param_groups[0]['lr']
            print(f"Step {step+1:4d}/{steps} | Loss: {loss.item():.3f} | Recall: {recall:.1%} | Prec: {prec:.1%} | LR: {lr:.5f}")
            if recall > best_recall:
                best_recall = recall
                torch.save({"model": model.state_dict(), "config": vars(cfg), "classes": ["face"], "recall": recall}, "runs/faces/nanostream_faces.pt")
            model.train()
        step += 1

print(f"\nTraining complete! Best Recall: {best_recall:.1%}")

## 🧪 Step 4: Run High-Level Detection & Visual Verification

In [ ]:
import nanostream, glob
import matplotlib.pyplot as plt

model = nanostream.load_model("runs/faces/nanostream_faces.pt", device=device.type)
test_imgs = glob.glob("data/real_faces/images/*.png")[:4]

plt.figure(figsize=(16, 4))
for idx, img_p in enumerate(test_imgs):
    dets = nanostream.detect(model, img_p, conf_thr=0.30)
    _, bgr = nanostream.preprocess_image(img_p)
    vis = nanostream.draw_detections(bgr, dets)
    
    plt.subplot(1, 4, idx + 1)
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title(f"{len(dets)} Face(s) Found")
    plt.axis("off")
plt.tight_layout()
plt.show()

## ⚡ Step 5: Export to Static Microcontroller C-Header (`model_weights.h`)

In [ ]:
!python -m nanostream.export_cli --model runs/faces/nanostream_faces.pt --out nanostream/mcu/model_weights.h

print("C Header ready for ESP32-S3 / ARM Cortex-M4!")
from google.colab import files
files.download("nanostream/mcu/model_weights.h")
files.download("runs/faces/nanostream_faces.pt")